# T10 - Decorators Test

Covers course section **10 - Python Decorators** (folder `10-Python Decorators`) and these course notebooks:

- `01-Decorators.ipynb`
- `02-Decorators Homework.ipynb` (optional homework, background only)

**14 questions / 20 points** (4 multiple choice x 1 point, 4 conceptual x 1 point, 6 coding x 2 points).

Suggested time budget: **45 minutes**.

How the checks work: each coding exercise is followed by a code cell tagged `check`. Run that cell after you write your answer; it asserts the expected results and prints nothing when your answer is correct (an `AssertionError` means the answer is still wrong).

See `S0 - Testing Syllabus.ipynb` for the full testing scheme and the other sections of the course.

## Q1. Multiple choice

In the course example, `func_needs_decorator = new_decorator(func_needs_decorator)` is executed.
After that line, what does the name `func_needs_decorator` refer to?

A. The original function, unchanged
B. The wrapper function `wrap_func` that `new_decorator` returned
C. The name `new_decorator`
D. `None`, because the original function was deleted

<details>
<summary>Solution</summary>

**Answer: B** - the wrapper function `wrap_func` that `new_decorator` returned.

`new_decorator(func_needs_decorator)` calls the decorator, which returns `wrap_func`, and that
object is then bound to the name `func_needs_decorator`.

**Why:** the same thing happens with the `@` syntax. `@new_decorator` above a `def` is exactly
shorthand for `func_name = new_decorator(func_name)` - the name ends up pointing at the wrapper,
not at the original function.

</details>

## Q2. Multiple choice

Which statement about closures is correct?

A. A nested function can only use variables defined in its own body
B. A nested function keeps access to the enclosing function's variables even after the enclosing function has returned
C. A nested function can never modify a variable of the enclosing function
D. Closures only work for functions that are decorated

<details>
<summary>Solution</summary>

**Answer: B** - a nested function keeps access to the enclosing function's variables even
after the enclosing function has returned.

**Why:** this is why `hello()` in the course notebook can return `greet` and that returned
function still works when called later. If the nested function needs to *assign* to an enclosing
variable it must declare it `nonlocal`.

</details>

## Q3. Multiple choice

What is the purpose of `functools.wraps`?

A. It makes the wrapper run faster
B. It copies metadata such as `__name__` and `__doc__` from the wrapped function onto the wrapper
C. It automatically converts the wrapper into a generator
D. It allows the wrapper to accept any arguments

<details>
<summary>Solution</summary>

**Answer: B** - it copies metadata such as `__name__` and `__doc__` from the wrapped
function onto the wrapper.

**Why:** without `functools.wraps`, a decorated function reports the wrapper's name
(`wrapper`) and loses its docstring, which makes debugging and introspection confusing.

</details>

## Q4. Multiple choice

You want a decorator that can be used with arguments, like `@repeat(3)`. How must such a
decorator be structured?

A. It must be a decorator factory: a function that takes the arguments and returns the actual decorator
B. It must use `*args` and `**kwargs` only
C. It must be a class with a `__call__` method
D. It must call `functools.wraps` inside

<details>
<summary>Solution</summary>

**Answer: A** - it must be a decorator factory: a function that takes the arguments and
returns the actual decorator.

**Why:** `@repeat(3)` is evaluated as `repeat(3)`, so `repeat(3)` must return a decorator, and that
decorator is then applied to the function. `*args`/`**kwargs` solve a different problem (accepting
any call signature), and `functools.wraps` is about metadata only.

</details>

## Q5. Conceptual - predict the output

What does this code print, and what is `x`?

```text
def hello(name='Jose'):

    def greet():
        return 'This is inside the greet() function'

    def welcome():
        return 'This is inside the welcome() function'

    if name == 'Jose':
        return greet
    else:
        return welcome

x = hello()
print(x)
print(x())
```

<details>
<summary>Solution</summary>

**Output:**

```text
<function hello.<locals>.greet at 0x...>
This is inside the greet() function
```

`x` is the `greet` function object defined inside `hello`.

**Why:** `hello()` returns `greet` (no parentheses), so the function object is returned instead of
its result. The second `print` calls that returned function, which is the closure over
`hello`'s scope.

</details>

## Q6. Conceptual - what does the decorator do

Given:

```text
def new_decorator(func):

    def wrap_func():
        print('Code would be here, before executing the func')
        func()
        print('Code here will execute after the func()')

    return wrap_func
```

and

```text
@new_decorator
def func_needs_decorator():
    print('This function is in need of a Decorator')
```

what is printed when `func_needs_decorator()` is called, and in what order?

<details>
<summary>Solution</summary>

**Output:**

```text
Code would be here, before executing the func
This function is in need of a Decorator
Code here will execute after the func()
```

**Why:** `@new_decorator` rebinds the name `func_needs_decorator` to `wrap_func`, so calling the
name calls the wrapper. The wrapper runs its first `print`, then calls the original function
object, then runs its second `print`.

</details>

## Q7. Conceptual - order of multiple decorators

Given two decorators `first` and `second` and:

```text
@first
@second
def work():
    print('work')
```

In which order are the decorators applied, and in which order does the printing happen when
`work()` is called?

A. `first` is applied first, then `second`; printing order is `work`, then the `second` code, then the `first` code
B. `second` is applied first, then `first`; printing order is the `second` code, then `work`, then the `first` code
C. `first` is applied first, then `second`; printing order is the `first` code, then `work`, then the `second` code
D. Both decorators are applied at the same time, so the order is undefined

<details>
<summary>Solution</summary>

**Answer: B** - `second` is applied first, then `first`; the printing order is the
`second` code, then `work`, then the `first` code.

**Why:** decorators are applied from the bottom up, so `work` becomes `second(work)` and then
`first(second(work))`. At call time the outermost wrapper (`first`) runs first, calls the inner
wrapper (`second`), which calls the original `work`.

</details>

## Q8. Conceptual - why `*args` and `**kwargs`

A decorator is written like this:

```text
def my_decorator(func):
    def wrapper(*args, **kwargs):
        print('calling', func.__name__)
        return func(*args, **kwargs)
    return wrapper
```

What would go wrong if `wrapper` were written as `def wrapper():` instead, and why does
`*args`/`**kwargs` fix it?

<details>
<summary>Solution</summary>

**Without `*args`/`**kwargs`** the wrapper accepts no arguments at all, so
decorating a function that takes arguments raises a `TypeError` as soon as it is called, because
Python cannot pass the arguments through.

**Why:** `*args` collects any positional arguments into a tuple and `**kwargs` collects any
keyword arguments into a dictionary, so the wrapper can forward them unchanged with
`func(*args, **kwargs)`. This is what lets one decorator work with functions of any signature.

</details>

## Q9. Coding - a closure with `nonlocal`

Write a function `make_counter()` that returns a function `inc()`.

Each time `inc()` is called it increases an internal counter by one and returns the new counter
value. The counter must live in the enclosing `make_counter` scope, so use `nonlocal`.

In [ ]:
# your answer here

<details>
<summary>Solution</summary>

```python
def make_counter():
    count = 0

    def inc():
        nonlocal count
        count += 1
        return count

    return inc
```

**Why:** `nonlocal count` tells Python to use the `count` variable from the enclosing function instead of creating a new local one.

</details>

In [ ]:
counter = make_counter()
assert [counter(), counter(), counter()] == [1, 2, 3]
other = make_counter()
assert other() == 1

## Q10. Coding - your first decorator

Write a decorator called `shout` that wraps a function and returns its result in upper case.

Use it like this:

```text
@shout
def greet(name):
    return 'hello ' + name
```

so that `greet('jose')` returns `'HELLO JOSE'`.

In [ ]:
# your answer here

<details>
<summary>Solution</summary>

```python
def shout(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs).upper()

    return wrapper
```

**Why:** The decorator receives the function and returns a wrapper that calls the original and upper-cases the result.

</details>

In [ ]:
@shout
def greet(name):
    return 'hello ' + name

assert greet('jose') == 'HELLO JOSE'
assert shout(lambda: 'quiet')() == 'QUIET'

## Q11. Coding - a decorator that works with any signature

Write a decorator called `pass_through` that calls the wrapped function and returns its result
unchanged, but works no matter how many arguments the wrapped function takes.

Use `*args` and `**kwargs` inside the wrapper so that arguments are forwarded correctly.

In [ ]:
# your answer here

<details>
<summary>Solution</summary>

```python
def pass_through(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)

    return wrapper
```

**Why:** `*args` and `**kwargs` collect every argument and `func(*args, **kwargs)` forwards them unchanged.

</details>

In [ ]:
@pass_through
def add(a, b):
    return a + b

@pass_through
def greet(name, greeting='hello'):
    return greeting + ' ' + name

@pass_through
def no_args():
    return 'nothing'

assert add(2, 3) == 5
assert greet('jose') == 'hello jose'
assert greet('jose', greeting='hi') == 'hi jose'

## Q12. Coding - preserving metadata with `functools.wraps`

Write a decorator called `preserve` that calls the wrapped function and returns its result, and
that uses `functools.wraps` so the decorated function keeps its original `__name__` and
`__doc__`.

Test it like this:

```text
@preserve
def documented(a, b):
    'Adds two numbers.'
    return a + b
```

In [ ]:
# your answer here

<details>
<summary>Solution</summary>

```python
import functools

def preserve(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)

    return wrapper
```

**Why:** `functools.wraps(func)` copies `__name__`, `__doc__` and friends from `func` onto the wrapper.

</details>

In [ ]:
@preserve
def documented(a, b):
    'Adds two numbers.'
    return a + b

assert documented(2, 3) == 5
assert documented.__name__ == 'documented'
assert documented.__doc__ == 'Adds two numbers.'

## Q13. Coding - a decorator factory

Write a decorator factory called `repeat` that takes the number of times to call a function and
returns a decorator. The decorated function is called that many times and the decorator returns
the list of all results.

Use it like this:

```text
@repeat(3)
def say_hi():
    return 'hi'
```

so that `say_hi()` returns `['hi', 'hi', 'hi']`.

In [ ]:
# your answer here

<details>
<summary>Solution</summary>

```python
def repeat(times):
    def decorator(func):
        def wrapper(*args, **kwargs):
            return [func(*args, **kwargs) for _ in range(times)]

        return wrapper

    return decorator
```

**Why:** `repeat(times)` returns the decorator, and that decorator returns the wrapper which calls the function `times` times.

</details>

In [ ]:
@repeat(3)
def say_hi():
    return 'hi'

assert say_hi() == ['hi', 'hi', 'hi']

@repeat(2)
def add(a, b):
    return a + b

assert add(1, 2) == [3, 3]
assert repeat(1)(lambda: 'once')() == ['once']

## Q14. Coding - a timing decorator

Write a decorator called `timed` that calls the wrapped function, measures how long the call took
using `time.perf_counter()`, and stores the elapsed seconds on the wrapper as the attribute
`elapsed`.

The decorated function must still return the original result, and `timed(func).elapsed` must be
the number of seconds the call took (a float of zero or more).

In [ ]:
# your answer here

<details>
<summary>Solution</summary>

```python
import time

def timed(func):
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        wrapper.elapsed = time.perf_counter() - start
        return result

    wrapper.elapsed = 0.0
    return wrapper
```

**Why:** `time.perf_counter()` is the monotonic clock; the difference between the two readings is the elapsed time, which is stored on the wrapper.

</details>

In [ ]:
@timed
def add(a, b):
    return a + b

assert add(2, 3) == 5
assert add.elapsed >= 0

@timed
def slow_square(n):
    total = 0
    for i in range(n):
        total += i * i
    return total

assert slow_square(4) == 14

## Topics covered in this test

- Functions as objects - assigning, passing and deleting names that hold functions
- Functions within functions and returning functions from functions
- Closures and `nonlocal`
- The decorator pattern: `func = new_decorator(func)` and the `@` syntax
- What the decorator syntax does to the name of the decorated function
- `*args` and `**kwargs` inside a decorator so it accepts any call signature
- `functools.wraps` and preserving `__name__` / `__doc__`
- Decorator factories - decorators that take arguments
- Stacking multiple decorators and the order of application
- Timing and logging decorator patterns